# Hung Sprint 2 — UrbanEV Development smoke (Colab)

This notebook orchestrates versioned repository CLIs; it contains no model logic. It is **unexecuted preparation**. The first run is a bounded 7-day Source/Development smoke across one 14-day replay block boundary, not full Source pretraining, parameter selection, effectiveness evidence, or Final evaluation. The runner is CPU-only. Record any changed smoke settings and keep all large outputs in a new Drive folder.

Prerequisites: put the published `inputs-urbanev-SZH-v1.zip` in your Drive, set the two paths below, and use a fresh Colab runtime. The download is independently checked against the fixed release catalog before extraction. The selected Development ID `1001` has 15 created origins in the verified local 7-day schedule; this is an engineering smoke target, not a performance-selected target.

For a separate source_checkpoint_only smoke, rerun in a fresh runtime with ENTITY_ID set to `1025` and a new RUN_ID. The verified 7-day Development inventory gives this ID zero initial target examples and 15 created origins; target-smoke should record zero adaptation steps. Keep this stratum separate from ID `1001`. This variant has not been executed.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CODE_SHA = "e609f0bf8cec53912bc4c9979414ddacb0eff209"
ARCHIVE_ON_DRIVE = Path("/content/drive/MyDrive/CHANGE_ME/inputs-urbanev-SZH-v1.zip")
DRIVE_RUN_PARENT = Path("/content/drive/MyDrive/CHANGE_ME/hung-sprint2-runs")
RUN_ID = "hung-urbanev-7d-smoke-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SEED = 7
ENTITY_ID = "1001"
SMOKE = {
    "source_steps": 2,
    "target_steps": 2,
    "batch_size": 2,
    "learning_rate": 0.001,
    "retention_weight": 0.1,
    "epsilon": 1e-6,
    "max_origins": 15,
    "level_one": 0.1,
    "level_two": 0.2,
    "initial_level": 0.0,
    "block_days": 14,
    "min_complete_days": 5,
    "delta": 0.0,
    "alpha": 0.05,
    "bootstrap_segment_days": 2,
    "bootstrap_resamples": 100,
    "challenger_rule": "bonferroni_two",
}
assert "CHANGE_ME" not in str(ARCHIVE_ON_DRIVE) + str(DRIVE_RUN_PARENT)
assert ARCHIVE_ON_DRIVE.is_file()
print("run_id", RUN_ID)

In [ ]:
repo = Path("/content/hung-sprint2-code")
if repo.exists():
    raise FileExistsError(repo)
subprocess.run(
    ["git", "clone", "https://github.com/HungLD9505/adaptive-retention-ev.git", str(repo)],
    check=True,
)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", CODE_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
assert actual_sha == CODE_SHA
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-dev.txt")], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
        "-r",
        str(repo / "requirements-model.txt"),
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("code_sha", actual_sha)


def run_cli(*args):
    command = [sys.executable, "-m", "experiments.hung_sprint2.cli", *map(str, args)]
    print("COMMAND:", " ".join(command))
    subprocess.run(command, cwd=repo, check=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_drive_run(folder):
    folder = Path(folder)
    marker = folder / "handoff_complete.json"
    if not marker.is_file():
        raise ValueError(f"Drive handoff is incomplete: {folder}")
    recorded = json.loads(marker.read_text(encoding="utf-8"))
    if (
        recorded.get("schema") != "hung-sprint2-drive-handoff-v1"
        or recorded.get("code_sha") != CODE_SHA
    ):
        raise ValueError("Unexpected Drive handoff schema/code")
    actual = {
        p.relative_to(folder).as_posix() for p in folder.rglob("*") if p.is_file() and p != marker
    }
    if actual != set(recorded["files"]):
        raise ValueError("Drive handoff file set differs")
    for name, expected in recorded["files"].items():
        if sha256_file(folder / name) != expected:
            raise ValueError(f"Drive handoff hash mismatch: {name}")
    return recorded


def publish_run(local_folder, drive_folder):
    local_folder = Path(local_folder)
    drive_folder = Path(drive_folder)
    drive_folder.parent.mkdir(parents=True, exist_ok=True)
    if drive_folder.exists():
        raise FileExistsError(drive_folder)
    shutil.copytree(local_folder, drive_folder)
    files = {}
    for original in local_folder.rglob("*"):
        if original.is_file():
            name = original.relative_to(local_folder).as_posix()
            digest = sha256_file(original)
            if digest != sha256_file(drive_folder / name):
                raise ValueError(f"Drive copy mismatch: {name}")
            files[name] = digest
    marker = {
        "schema": "hung-sprint2-drive-handoff-v1",
        "code_sha": CODE_SHA,
        "files": files,
    }
    with (drive_folder / "handoff_complete.json").open("x", encoding="utf-8") as stream:
        json.dump(marker, stream, indent=2, sort_keys=True)
        stream.write("\n")
    verify_drive_run(drive_folder)
    return drive_folder

In [ ]:
archive = Path("/content/inputs-urbanev-SZH-v1.zip")
input_root = Path("/content/hung-sprint2-data")
if archive.exists() or input_root.exists():
    raise FileExistsError("Use a fresh archive/input destination")
shutil.copyfile(ARCHIVE_ON_DRIVE, archive)
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(archive),
        "--package",
        archive.name,
    ],
    cwd=repo,
    check=True,
)
with zipfile.ZipFile(archive) as package:
    names = package.namelist()
    prefix = "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-"
    if len(names) != len(set(names)) or any(
        not name.startswith(prefix) or Path(name).is_absolute() or ".." in Path(name).parts
        for name in names
    ):
        raise ValueError("Unexpected archive entry")
    input_root.mkdir()
    package.extractall(input_root)
run = input_root / "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-7d"
manifest_sha = hashlib.sha256((run / "manifest.json").read_bytes()).hexdigest()
print("input_run", run.name, "manifest_sha256", manifest_sha)

In [ ]:
run_root = Path("/content/hung-sprint2-runs") / RUN_ID
if run_root.exists():
    raise FileExistsError(run_root)
run_root.mkdir(parents=True)


source_out = run_root / "source"
target_out = run_root / "target"
replay_out = run_root / "replay-15"
run_cli(
    "source-smoke",
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--output",
    source_out,
    "--steps",
    SMOKE["source_steps"],
    "--batch-size",
    SMOKE["batch_size"],
    "--seed",
    SEED,
    "--learning-rate",
    SMOKE["learning_rate"],
    "--calendar-hidden",
    32,
)
run_cli(
    "target-smoke",
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    source_out / "source_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    target_out,
    "--steps",
    SMOKE["target_steps"],
    "--batch-size",
    SMOKE["batch_size"],
    "--seed",
    SEED,
    "--learning-rate",
    SMOKE["learning_rate"],
    "--retention-weight",
    SMOKE["retention_weight"],
    "--epsilon",
    SMOKE["epsilon"],
)
replay_args = [
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    source_out / "source_checkpoint.pt",
    "--target-checkpoint",
    target_out / "target_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    replay_out,
    "--max-origins",
    SMOKE["max_origins"],
    "--block-days",
    SMOKE["block_days"],
    "--min-complete-days",
    SMOKE["min_complete_days"],
    "--bootstrap-segment-days",
    SMOKE["bootstrap_segment_days"],
    "--bootstrap-resamples",
    SMOKE["bootstrap_resamples"],
    "--seed",
    SEED,
    "--level-one",
    SMOKE["level_one"],
    "--level-two",
    SMOKE["level_two"],
    "--initial-level",
    SMOKE["initial_level"],
    "--delta",
    SMOKE["delta"],
    "--alpha",
    SMOKE["alpha"],
    "--epsilon",
    SMOKE["epsilon"],
    "--challenger-rule",
    SMOKE["challenger_rule"],
]
run_cli("replay-development", *replay_args)
summary = json.loads((replay_out / "run_manifest.json").read_text())
assert summary["runner_cursor"] == SMOKE["max_origins"]
print("coverage", summary["coverage"], "metrics_kwh", summary["metrics_kwh"]["live"])

In [ ]:
# Publish only after checking the copied files; an interrupted folder has no completion marker.
drive_run = publish_run(run_root, DRIVE_RUN_PARENT / RUN_ID)
print("drive_run", drive_run)
print("runner_checkpoint_sha256", summary["final_runner_checkpoint_sha256"])

## Resume on a fresh runtime

Run the setup and verified extraction cells again. Set `PREVIOUS_RUN_ID` to the published Drive run ID, then execute the cell below. It reads the previous Source/target and runner checkpoints from Drive, verifies lineage/config/code through `replay-development`, creates a **new** local output, and copies that new output to a separate Drive folder. It does not reissue earlier forecasts. The smoke settings above are examples for technical validation on Development and are not locked research choices.


In [ ]:
PREVIOUS_RUN_ID = "CHANGE_ME_PREVIOUS_RUN_ID"
assert PREVIOUS_RUN_ID != "CHANGE_ME_PREVIOUS_RUN_ID"
previous = DRIVE_RUN_PARENT / PREVIOUS_RUN_ID
verify_drive_run(previous)
assert (previous / "replay-15/runner_checkpoint.pt").is_file()
resume_out = Path("/content/hung-sprint2-resume") / (PREVIOUS_RUN_ID + "-to-29")
if resume_out.exists():
    raise FileExistsError(resume_out)
resume_args = [
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    previous / "source/source_checkpoint.pt",
    "--target-checkpoint",
    previous / "target/target_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    resume_out,
    "--max-origins",
    29,
    "--block-days",
    SMOKE["block_days"],
    "--min-complete-days",
    SMOKE["min_complete_days"],
    "--bootstrap-segment-days",
    SMOKE["bootstrap_segment_days"],
    "--bootstrap-resamples",
    SMOKE["bootstrap_resamples"],
    "--seed",
    SEED,
    "--level-one",
    SMOKE["level_one"],
    "--level-two",
    SMOKE["level_two"],
    "--initial-level",
    SMOKE["initial_level"],
    "--delta",
    SMOKE["delta"],
    "--alpha",
    SMOKE["alpha"],
    "--epsilon",
    SMOKE["epsilon"],
    "--challenger-rule",
    SMOKE["challenger_rule"],
]
run_cli(
    "replay-development",
    *resume_args,
    "--resume-checkpoint",
    previous / "replay-15/runner_checkpoint.pt",
)
new_drive_run = publish_run(resume_out, DRIVE_RUN_PARENT / (PREVIOUS_RUN_ID + "-resume-29"))
print("resumed_drive_run", new_drive_run)